In [ ]:
#
# Import Landsat Data and Calculate NDVI
#

import rasterio
import os
import geopandas as gpd
import xarray as xr
import rioxarray
import numpy as np
import matplotlib.pyplot as plt
import gdown
from scipy.stats import linregress
import cmcrameri.cm as cmc
from matplotlib.colors import LightSource

# Create data directory
data_folder = "data/raw"
os.makedirs(data_folder, exist_ok=True)


quart_fp = "data/raw/stzh.adm_statistische_quartiere_map.json"
quart = gpd.read_file(quart_fp)

filepath = "data/raw/LandsatComposite_Zurich_1985.tif"

# Load test data cube
#ds = xr.tutorial.open_dataset("air_temperature")
#print(ds)

# Loop over years to create filepaths filepaths
years = [1985, 1988, 1991, 1994, 1997, 2000, 2003, 2006, 2009, 2012, 2015, 2018, 2021, 2024]

ls_paths = []

for year in years:
	fp = "data/raw/LandsatComposite_Zurich_" + str(year) + ".tif"

	ls_paths.append(fp)

#print(ls_paths)

# Loop over filepaths to import data into data cube
list_ds = []

for path in ls_paths:
	da = rioxarray.open_rasterio(path)

	dt = int(path.split("_")[2].split(".")[0])

	ds = xr.Dataset(
		{
			"Blue": da.sel(band = 1).drop_vars("band", errors="ignore"), 
			"Green": da.sel(band = 2).drop_vars("band", errors="ignore"), 
			"Red": da.sel(band = 3).drop_vars("band", errors="ignore"), 
			"NIR": da.sel(band = 4).drop_vars("band", errors="ignore"), 
			"SWIR1":  da.sel(band = 5).drop_vars("band", errors="ignore"), 
			"SWIR2": da.sel(band = 6).drop_vars("band", errors="ignore"), 
			"TIR": da.sel(band = 7).drop_vars("band", errors="ignore"),
		}
	)

	#ds = ds.assign_coords(band = bands)
	ds = ds.expand_dims(time = [dt])
	ds["NDVI"] = (ds.NIR - ds.Red) / (ds.NIR + ds.Red)
	ds["LST"] = ds.TIR - 273.15

	list_ds.append(ds)

ls_cube = xr.combine_by_coords(list_ds)

# Open the raster connection safely
#with rasterio.open(filepath) as src:
	#print(src.name)
	#print(src.mode)
	#print("Band count:", src.count)

# Bands
# Blue, Green, Red, NIR, SWIR1, SWIR2, TIR

In [ ]:
#
# Calculate NDVI
#

import rasterio
import numpy as np
import matplotlib.pyplot as plt

with rasterio.open(filepath) as src:
	# Read the bands and immediately convert them to float to allow division
	# Band 3 = Red, Band 4 = NIR
	red = src.read(3).astype(float)
	nir = src.read(4).astype(float)

# Suppress warnings for zero division and invalid operations
with np.errstate(divide="ignore", invalid="ignore"):
	# Perform the pixel by pixel array math
	ndvi = (nir - red) / (nir + red)

# Plot the calculated index
plt.figure(figsize=(7, 6))
plt.imshow(ndvi)
plt.title("NDVI")
plt.axis("off")
plt.show()

In [ ]:
#
# Visualise RGB image
#

import numpy as np
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(filepath) as src:
	red = src.read(3)
	green = src.read(2)
	blue = src.read(1)


def normalize(array, vmin=0, vmax=0.4):
	"""Normalize and clip array to a specific range (default 0 to 0.4)"""
	# Clip the array to the specified vmin and vmax
	clipped = np.clip(array, vmin, vmax)
	# Scale the clipped array to 0-1 for display
	return (clipped - vmin) / (vmax - vmin)


red_n = normalize(red)
green_n = normalize(green)
blue_n = normalize(blue)

# Stack the normalized bands along a third dimension
rgb = np.dstack((red_n, green_n, blue_n))

plt.figure(figsize=(7, 6))
plt.imshow(rgb)
plt.title("RGB true color composite (Scaled 0-0.4)")
plt.axis("off")
plt.show()

In [ ]:
#
# Visualise IR Bands
#

import matplotlib.pyplot as plt
from rasterio.plot import show

with rasterio.open(filepath) as src:
	fig, (ax4, ax5, ax6, ax7) = plt.subplots(ncols=4, figsize=(11, 4), sharey=True)
#, ax4, ax5, ax6, ax7
	#show((src, 1), cmap="viridis", ax=ax1)
	#show((src, 2), cmap="viridis", ax=ax2)
	#show((src, 3), cmap="viridis", ax=ax3)
	show((src, 4), cmap="viridis", ax=ax4)
	show((src, 5), cmap="viridis", ax=ax5)
	show((src, 6), cmap="viridis", ax=ax6)
	show((src, 7), cmap="viridis", ax=ax7)

	#ax1.set_title("Band 1 (Blue)")
	#ax2.set_title("Band 2 (Green)")
	#ax3.set_title("Band 3 (Red)")
	ax4.set_title("Band 4 (NIR)")
	ax5.set_title("Band 5 (SWIR1)")
	ax6.set_title("Band 6 (SWIR2)")
	ax7.set_title("Band 7 (TIR)")
	
	#ax1.axis("off")
	#ax2.axis("off")
	#ax3.axis("off")
	ax4.axis("off")
	ax5.axis("off")
	ax6.axis("off")
	ax7.axis("off")

plt.show()

In [ ]:
#
# Visualise NDVI
#

ndvi_fg = ls_cube.NDVI.isel(time=slice(0,1)).plot(
	col="time", col_wrap=1, robust=True, cbar_kwargs={"label": "NDVI"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.axis("off")
plt.show()

In [ ]:
#
# Decadal Trend Temperature
#

# 1. Apply linear regression across the time dimension for the entire cube
trends = ls_cube.polyfit(dim="time", deg=1)

# 2. Extract and scale slope (degree=1 is change per year, * 10 for change per decade)
trend_map = trends.LST_polyfit_coefficients.sel(degree=1) * 10

# 4. Plot the regional trend map
fig, ax = plt.subplots()

# Use RdBu_r so red indicates a negative trend (snow loss)
trend_map.plot(
	ax=ax, 
	cmap='RdBu_r', 
	#vmin=-0.08,
	#vmax=0.08,
	cbar_kwargs={'label': 'Temperature Change per Decade (1985-2024)'}
)

ax.set_title("Decadal Temperature Trend in (°C)")
plt.legend(loc='lower right')
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#
# Decadal Trend in NDVI
#

# 1. Apply linear regression across the time dimension for the entire cube
trends = ls_cube.polyfit(dim="time", deg=1)

# 2. Extract and scale slope (degree=1 is change per year, * 10 for change per decade)
trend_map = trends.NDVI_polyfit_coefficients.sel(degree=1) * 10

# 4. Plot the regional trend map
fig, ax = plt.subplots()

# Use RdBu_r so red indicates a negative trend (snow loss)
trend_map.plot(
	ax=ax, 
	cmap='RdBu', 
	#vmin=-0.08,
	#vmax=0.08,
	cbar_kwargs={'label': 'NDVI Change per Decade'}
)

ax.set_title("Decadal Trend in NDVI (1985-2024)")
plt.legend(loc='lower right')
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
#
# Visualise Temperature Difference
#

tir_1985 = ls_cube.TIR.sel(time=1985)
tir_2024 = ls_cube.TIR.sel(time=2024)

tir_diff = tir_2024 - tir_1985

# Create the 1x3 subplot comparison
fig, axes = plt.subplots(3, 1)

# Plot 2019 FCC
axes[0].imshow(tir_1985)
axes[0].set_title("1985 TIR")
axes[0].axis("off")

# Plot 2023 FCC
axes[1].imshow(tir_2024)
axes[1].set_title("2024 TIR")
axes[1].axis("off")

# Plot EVI Change using a diverging colormap
im = axes[2].imshow(tir_diff, cmap=cmc.bam)
axes[2].set_title("Temperature Change (1985 to 2024)")
axes[2].axis("off")

# Add a colorbar for the EVI change plot
cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("Change in K")

plt.tight_layout()
plt.show()

In [ ]:

# LST Difference

lst_1985 = ls_cube.LST.sel(time=1985)
lst_2024 = ls_cube.LST.sel(time=2024)

lst_diff = lst_2024 - lst_1985

lst_diff_fg = lst_diff.plot(
	col_wrap=1, robust=True, cbar_kwargs={"label": "Temperature Difference (°C)"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.title('Temperature Difference (1985-2024)')
plt.axis("off")
plt.show()

# NDVI Difference

ndvi_1985 = ls_cube.NDVI.sel(time=1985)
ndvi_2024 = ls_cube.NDVI.sel(time=2024)

ndvi_diff = ndvi_2024 - ndvi_1985

ndvi_diff_fg = ndvi_diff.plot(
	col_wrap=1, robust=True, cbar_kwargs={"label": "NDVI Difference"}
)

ndvi_fg.map_dataarray(
	xr.plot.contour,
	x="x",
	y="y",
	colors="k",
	levels=5,
	linewidths=0.5,
	add_colorbar=False,
)
# Turn into subplot to turn off axes
plt.title('Vegetation Difference (1985-2024)')
plt.axis("off")
plt.show()